# Test Metrics Summary



This notebook reads a `test_metrics_over_all.csv` file and prints mean/std per metric, overall and by body part.

In [6]:
from pathlib import Path
import pandas as pd

# Update this path to your CSV
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

# The file has a header row: file_name, MAE, MSE, PSNR, SSIM
# Some files may include a leading unnamed index column; drop it if present.
df = pd.read_csv(csv_path)

if "file_name" not in df.columns and df.shape[1] >= 6:
    df = df.iloc[:, 1:]
    df.columns = ["file_name", "MAE", "MSE", "PSNR", "SSIM"]

# Convert metrics to numeric (invalid values become NaN)
for col in ["MAE", "MSE", "PSNR", "SSIM"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()


,Unnamed: 0,file_name,MAE,MSE,PSNR,SSIM
0,0,AB_1ABA009-1.nii,NaN,NaN,NaN,-0.999876
1,1,AB_1ABA009-10.nii,248.336511,403.156389,46.190343,0.001814
2,2,AB_1ABA009-100.nii,2007.000000,30353.000000,27.423062,-0.002931
3,3,AB_1ABA009-101.nii,2007.000000,30353.000000,27.423062,-0.010321
4,4,AB_1ABA009-102.nii,NaN,NaN,NaN,-0.999876


In [7]:
# Overall mean and std formatted as mean ± std
metrics = ["MAE", "MSE", "PSNR", "SSIM"]

overall_mean = df[metrics].mean()
overall_std = df[metrics].std()

overall_summary = pd.DataFrame({
    "mean": overall_mean,
    "std": overall_std,
})

formatted_overall = overall_summary.apply(
    lambda row: f"{row['mean']:.6g} ± {row['std']:.6g}",
    axis=1,
)

formatted_overall = formatted_overall.to_frame(name="mean ± std")

print("Overall metrics:")
formatted_overall


Overall metrics:


,mean ± std
MAE,181.106 ± 281.175
MSE,3235.65 ± 3931.77
PSNR,38.0827 ± 2.5478
SSIM,0.891837 ± 0.286712


In [8]:
# Body part grouping derived from filename prefix (e.g., AB_*, HN_*)
# Example filename: AB_1ABC127-99.nii -> body_part = AB

df["body_part"] = df["file_name"].str.split("_").str[0]

grouped = df.groupby("body_part")[metrics]

by_part_mean = grouped.mean()
by_part_std = grouped.std()

by_part_summary = pd.concat(
    {"mean": by_part_mean, "std": by_part_std},
    axis=1,
)

formatted_by_part = by_part_mean.copy()
for metric in metrics:
    formatted_by_part[metric] = (
        by_part_mean[metric].map(lambda v: f"{v:.6g}")
        + " ± "
        + by_part_std[metric].map(lambda v: f"{v:.6g}")
    )

print("Metrics by body part:")
formatted_by_part


Metrics by body part:


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,171.829 ± 325.449,4681.05 ± 4522.03,36.2439 ± 2.08769,0.888906 ± 0.325452
HN,262.587 ± 429.41,4131.62 ± 6086.67,37.6178 ± 3.03196,0.819731 ± 0.396058
TH,196.057 ± 337.632,4267.31 ± 4766.46,36.7409 ± 2.07286,0.856031 ± 0.376205
brain,176.887 ± 146.08,1626.01 ± 858.324,40.2962 ± 1.73059,0.912817 ± 0.172416
pelvis,106.113 ± 85.2909,2803.77 ± 545.858,37.8247 ± 0.832359,0.963707 ± 0.079589
